# Tarea 1 – Parte 2: Modelado y clasificación de noticias

En esta sección se comparan tres enfoques para clasificar titulares de noticias:

1. Zero-Shot Classification.
2. Etiquetado mediante un LLM.
3. Transformer con fine-tuning.

In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("tarea_1") if Path("tarea_1").exists() else Path(".")

ruta_datos = BASE_DIR / "datos" / "noticias.csv"

df = pd.read_csv(ruta_datos)

print("Dimensiones:", df.shape)
print("\nDistribución por categoría:")
print(df["etiqueta"].value_counts())

df.head()

Dimensiones: (400, 7)

Distribución por categoría:
etiqueta
politica              100
economia              100
deportes              100
ciencia_tecnologia    100
Name: count, dtype: int64


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,''Evacúen sus casas desde ya'': Keiko Fujimori...,La mandataria mencionó que el fenómeno está af...,La República,politica,https://larepublica.pe/politica/2026/10/16/eva...,2026-10-16 07:52:00,politica
1,Keiko Fujimori sobre limpieza del río Piura an...,"Además, optó por darle la responsabilidad a la...",La República,politica,https://larepublica.pe/politica/2026/10/17/kei...,2026-10-17 08:36:31,politica
2,Debate de pedido de facultades sigue entrampad...,La Comisión de Constitución del Senado todavía...,La República,politica,https://larepublica.pe/politica/2026/10/17/deb...,2026-10-17 07:14:35,politica
3,Fuerza Popular desconoce votación que pedía de...,Cuatro bancadas de oposición abandonaron la se...,La República,politica,https://larepublica.pe/politica/2026/10/15/fue...,2026-10-15 19:05:12,politica
4,Solicitan la destitución de Roberto Burneo por...,"La asociación civil Iniciativa Democrática, or...",La República,politica,https://larepublica.pe/politica/2026/10/16/sol...,2026-10-16 14:54:58,politica


In [2]:
muestra_zero_shot = (
    df.groupby("etiqueta", group_keys=False)
      .sample(n=10, random_state=42)
      .reset_index(drop=True)
)

print("Tamaño de la muestra:", len(muestra_zero_shot))
print(muestra_zero_shot["etiqueta"].value_counts())

Tamaño de la muestra: 40
etiqueta
ciencia_tecnologia    10
deportes              10
economia              10
politica              10
Name: count, dtype: int64


In [3]:
import transformers
import torch
import sklearn

print(transformers.__version__)
print(torch.__version__)
print(sklearn.__version__)

c:\Users\Alvar\anaconda3\envs\webscraping\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.20.1
2.14.1+cpu
1.9.1


## 1. Clasificación Zero-Shot

Se utiliza un modelo Zero-Shot para clasificar una muestra balanceada de 40 titulares, sin realizar un entrenamiento previo con el dataset de noticias.

In [4]:
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

modelo_zero_shot = pipeline(
    "zero-shot-classification",
    model="Recognai/bert-base-spanish-wwm-cased-xnli",
    device=-1
)

categorias_modelo = [
    "política",
    "economía",
    "deportes",
    "ciencia y tecnología"
]

mapeo_prediccion = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia y tecnología": "ciencia_tecnologia"
}

c:\Users\Alvar\anaconda3\envs\webscraping\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Alvar\.cache\huggingface\hub\models--Recognai--bert-base-spanish-wwm-cased-xnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\Alvar\anaconda3\envs\webscraping\Lib\site-packages\huggingface_hub\file_down

In [5]:
titular_prueba = muestra_zero_shot.loc[0, "titular"]

resultado = modelo_zero_shot(
    titular_prueba,
    candidate_labels=categorias_modelo,
    hypothesis_template="Este titular trata sobre {}."
)

print("Titular:", titular_prueba)
print("Etiqueta real:", muestra_zero_shot.loc[0, "etiqueta"])
print("Predicción:", resultado["labels"][0])
print("Confianza:", round(resultado["scores"][0], 3))

Titular: La humanidad roza sus límites: la población mundial alcanzaría los 12.400 millones en 2070, mientras científicos sugieren reducirla a 4.000 millones para 2200
Etiqueta real: ciencia_tecnologia
Predicción: ciencia y tecnología
Confianza: 0.737


In [6]:
predicciones = []

for titular in muestra_zero_shot["titular"]:
    resultado = modelo_zero_shot(
        titular,
        candidate_labels=categorias_modelo,
        hypothesis_template="Este titular trata sobre {}."
    )

    predicciones.append({
        "prediccion_zero_shot": mapeo_prediccion[resultado["labels"][0]],
        "confianza_zero_shot": resultado["scores"][0]
    })

df_predicciones = pd.DataFrame(predicciones)

muestra_zero_shot = pd.concat(
    [muestra_zero_shot, df_predicciones],
    axis=1
)

In [7]:
accuracy_zero = accuracy_score(
    muestra_zero_shot["etiqueta"],
    muestra_zero_shot["prediccion_zero_shot"]
)

f1_zero = f1_score(
    muestra_zero_shot["etiqueta"],
    muestra_zero_shot["prediccion_zero_shot"],
    average="macro"
)

print(f"Accuracy Zero-Shot: {accuracy_zero:.3f}")
print(f"F1 Macro Zero-Shot: {f1_zero:.3f}")

Accuracy Zero-Shot: 0.800
F1 Macro Zero-Shot: 0.791


### Revisión de errores del modelo Zero-Shot

Se identifican los titulares cuya categoría predicha no coincide con la etiqueta original para analizar los principales errores del modelo.

In [8]:
errores_zero_shot = muestra_zero_shot[
    muestra_zero_shot["etiqueta"] !=
    muestra_zero_shot["prediccion_zero_shot"]
][
    ["titular", "etiqueta", "prediccion_zero_shot", "confianza_zero_shot"]
]

print(f"Errores encontrados: {len(errores_zero_shot)}")
errores_zero_shot

Errores encontrados: 8


,titular,etiqueta,prediccion_zero_shot,confianza_zero_shot
14,Luis Carlos Arias Schreiber: “Si Alejandro Olm...,deportes,politica,0.808821
22,"Tras cuernos, palos, por Kurt Burneo",economia,deportes,0.714816
26,"BCRP mantiene tasa de referencia en 4,25% por ...",economia,politica,0.580930
27,Reinfo: formalización minera hasta 2028 exigir...,economia,politica,0.905171
28,Aeropuerto Jorge Chávez: LAP revela que aerolí...,economia,politica,0.349987
29,Inteligencia artificial para conseguir trabajo...,economia,ciencia_tecnologia,0.640933
30,Fugó cabecilla de la banda que mató a “China P...,politica,deportes,0.726629
37,Madre de Dios: Uno de los 3 abatidos en La Pam...,politica,deportes,0.351529


## 2. Etiquetado con LLM local

Se utiliza el modelo `qwen3:4b-instruct`, ejecutado localmente mediante Ollama, para clasificar los titulares del dataset completo en una de las cuatro categorías definidas.

In [9]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODELO_LLM = "qwen3:4b-instruct"

respuesta = requests.get("http://localhost:11434/api/tags")

print("Conexión con Ollama:", respuesta.status_code)

Conexión con Ollama: 200


In [10]:
def clasificar_con_llm(titular):
    prompt = f"""
Clasifica el siguiente titular de noticia en UNA sola de estas categorías:

- politica
- economia
- deportes
- ciencia_tecnologia

Titular:
"{titular}"

Devuelve únicamente un JSON con esta estructura:

{{
  "etiqueta": "categoria",
  "confianza": 0.00
}}

La confianza debe ser un número entre 0 y 1.
No agregues explicaciones.
"""

    payload = {
        "model": MODELO_LLM,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": 0
        }
    }

    respuesta = requests.post(
        OLLAMA_URL,
        json=payload,
        timeout=120
    )

    respuesta.raise_for_status()

    contenido = respuesta.json()["response"]
    resultado = json.loads(contenido)

    return resultado

In [11]:
titular_prueba = df.loc[0, "titular"]

resultado_llm = clasificar_con_llm(titular_prueba)

print("Titular:", titular_prueba)
print("Etiqueta original:", df.loc[0, "etiqueta"])
print("Respuesta LLM:", resultado_llm)

Titular: ''Evacúen sus casas desde ya'': Keiko Fujimori pide a población que vive cerca a ríos que ''alisten su mochila'' ante El Niño
Etiqueta original: politica
Respuesta LLM: {'etiqueta': 'politica', 'confianza': 0.99}
